# 01 Update US and Singapore Data

Optional stage: restart the kernel and Run All to refresh prices. Requires `yfinance` and internet access. `run_all.py` skips this stage unless `--update-data` is given.

Download nine US ETFs, ES3.SI, A35.SI, and SGD=X from 2011-10-01 with `auto_adjust=False`, then overwrite the two price files:

| File | Contents | Consumers |
|---|---|---|
| `data/raw/etf_daily.csv` | All tickers in the original yfinance multi-level header format | 02, 05, 06, 07, 08 |
| `data/raw/sti_daily.csv` | One row per ES3 trading day; ES3, A35 (`bond_` prefix), and `usdsgd` | 03, 07, 08 |

- Existing files are first copied to `etf_daily_backup.csv` / `sti_daily_backup.csv` (each run replaces the previous backup).
- Today's row is dropped if downloaded before the local close (16:30 New York for US data, 17:30 Singapore for Singapore data), because it holds intraday prices.
- If any US ETF fails to download, neither file is overwritten. If only the Singapore tickers fail, `sti_daily.csv` is left unchanged.


In [ ]:
import sys
from pathlib import Path

# Locate the project root from the root itself or any subfolder, then share 00_engine via portfolio_accounting.
ROOT = Path.cwd().resolve()
while not (ROOT / 'pipeline' / '00_engine.ipynb').is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from portfolio_accounting import ENGINE as _engine, load_definitions

import shutil
import pandas as pd

TICKS = _engine['TICKS']


In [ ]:
DATA_DIR = ROOT / 'data' / 'raw'
ETF_CSV, STI_CSV = DATA_DIR / 'etf_daily.csv', DATA_DIR / 'sti_daily.csv'
SG = {'ES3': 'ES3.SI', 'A35': 'A35.SI', 'FX': 'SGD=X'}


def drop_unfinished(df, tz, close_minutes, label):
    """Drop today's row if it was downloaded before the local close (intraday prices)."""
    now = pd.Timestamp.now(tz=tz)
    if len(df) and df.index[-1].date() >= now.date() and now.hour * 60 + now.minute < close_minutes:
        print(f'{label}: dropped unfinished session {df.index[-1].date()} (local time {now:%H:%M}, market not yet closed)')
        df = df.iloc[:-1]
    return df


def has(data, t):
    return ('Adj Close', t) in data and not data['Adj Close'][t].dropna().empty


def download_all(start='2011-10-01'):
    import yfinance as yf
    data = yf.download(TICKS + list(SG.values()), start=start, auto_adjust=False)
    if data.empty:
        raise ValueError('Download returned no data; no files were overwritten.')
    missing = [t for t in TICKS if not has(data, t)]
    if missing:
        raise ValueError(f'Download is missing US tickers {missing}; no files were overwritten.')
    return data


def build_etf_daily(data):
    return drop_unfinished(data, 'America/New_York', 16 * 60 + 30, 'etf_daily (US equities)')


def build_sti_daily(data):
    """Same layout that 03 load_sti() expects: one row per ES3 trading day."""
    def ohlc(t, prefix=''):
        return pd.DataFrame({f'{prefix}open': data['Open'][t], f'{prefix}high': data['High'][t],
                             f'{prefix}low': data['Low'][t], f'{prefix}close': data['Close'][t],
                             f'{prefix}adj_close': data['Adj Close'][t], f'{prefix}volume': data['Volume'][t]})
    out = ohlc(SG['ES3']).dropna(subset=['close'])
    out = out.join(ohlc(SG['A35'], 'bond_').reindex(out.index))     # A35 gaps stay blank; loaders treat them as no new quote
    fx = data['Close'][SG['FX']].dropna()
    out['usdsgd'] = fx.reindex(out.index.union(fx.index)).ffill().reindex(out.index)   # latest FX quote on or before each date
    out.index.name = 'Date'
    return drop_unfinished(out, 'Asia/Singapore', 17 * 60 + 30, 'sti_daily (Singapore)')


def overwrite(df, target):
    if target.is_file():
        shutil.copy2(target, target.with_name(target.stem + '_backup.csv'))
    df.to_csv(target)


def main(download=True):
    if not download:
        print('Download skipped (DOWNLOAD = False).')
        return None
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    data = download_all()

    etf = build_etf_daily(data)
    overwrite(etf, ETF_CSV)
    last_us = etf['Adj Close'][TICKS].dropna().index.max().date()
    print(f'Updated {ETF_CSV.name} (previous copy: {ETF_CSV.stem}_backup.csv) | last complete US session: {last_us}')

    sti = None
    lost = [t for t in SG.values() if not has(data, t)]
    if lost:
        print(f'⚠ Singapore tickers {lost} failed to download; {STI_CSV.name} left unchanged.')
    else:
        sti = build_sti_daily(data)
        overwrite(sti, STI_CSV)
        print(f'Updated {STI_CSV.name} (previous copy: {STI_CSV.stem}_backup.csv) | '
              f'{sti.index[0].date()} ~ {sti.index[-1].date()} | A35 days with data: {int(sti["bond_close"].notna().sum())}')
    return {'etf_daily': etf, 'sti_daily': sti}


In [ ]:
DOWNLOAD = True      # set False to skip the download
results = main(download=DOWNLOAD)
